# EmotionCLIP-ReID RAF-DB Runbook

Notebook này dành riêng cho RAF-DB Basic. Protocol dùng official predefined split của RAF-DB: `train_*` để train và `test_*` để validation/test trong pipeline hiện tại. Không random split.

## 0. Kernel và repo

Chạy cell này trong đúng kernel/environment bạn muốn train.

In [10]:
from pathlib import Path
import os
import sys

REPO = Path.cwd()
if not (REPO / "train_emotionclip.py").exists():
    candidates = [Path("/mnt/e/Source/EmotionCLIP-ReID"), Path.home() / "EmotionCLIP-ReID"]
    for candidate in candidates:
        if (candidate / "train_emotionclip.py").exists():
            REPO = candidate
            os.chdir(REPO)
            break
print("Repo:", REPO)
print("Python:", sys.executable)

FileNotFoundError: [Errno 2] No such file or directory

## 1. Kiểm tra package tối thiểu

In [ ]:
import importlib.util
import subprocess
import sys

required = ["torch", "PIL", "yaml", "numpy","kaggle"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
print("Missing:", missing)
if missing:
    print("Cài environment theo environment_emotionclip_cuda.yml hoặc cài các package còn thiếu trước khi train.")

# Kaggle chỉ cần nếu bạn muốn tải bằng Kaggle thay vì upload archive RAF-DB sẵn.
print("Kaggle module:", importlib.util.find_spec("kaggle") is not None)

Missing: []
Kaggle module: True


## 2. Process RAF-DB đã có sẵn

Notebook dùng trực tiếp thư mục `data/RAF-DB` hiện có. Cell này chỉ convert `train_labels.csv` và `test_labels.csv` sang `manifest.jsonl` theo predefined RAF-DB Basic protocol. Không tải dữ liệu và không random split.


In [ ]:
!git pull


Already up to date.


In [11]:
from pathlib import Path
import subprocess
import sys

RAF_ROOT = REPO / "data" / "RAF-DB"
RAF_MANIFEST = RAF_ROOT / "manifest.jsonl"

if not RAF_ROOT.exists():
    raise FileNotFoundError(f"Không thấy RAF-DB tại {RAF_ROOT}")
for required in ["train_labels.csv", "test_labels.csv"]:
    if not (RAF_ROOT / required).exists():
        raise FileNotFoundError(f"Thiếu {RAF_ROOT / required}")

convert_cmd = [
    sys.executable,
    "tools/convert_rafdb_to_emotion_jsonl.py",
    "--raf-root",
    str(RAF_ROOT),
    "--output",
    str(RAF_MANIFEST),
    "--root-dir",
    str(RAF_ROOT),
]
print("Converting existing data/RAF-DB -> EmotionCLIP manifest")
subprocess.run(convert_cmd, cwd=REPO, check=True)
print("RAF-DB manifest ready:", RAF_MANIFEST)


Converting existing data/RAF-DB -> EmotionCLIP manifest


Traceback (most recent call last):
  File "/home/jupyter-hault/EmotionCLIP-ReID/tools/convert_rafdb_to_emotion_jsonl.py", line 256, in <module>
    main()
  File "/home/jupyter-hault/EmotionCLIP-ReID/tools/convert_rafdb_to_emotion_jsonl.py", line 242, in main
    label_file = Path(args.label_file).resolve() if args.label_file else _find_first_existing(raf_root, DEFAULT_LABEL_NAMES)
  File "/home/jupyter-hault/EmotionCLIP-ReID/tools/convert_rafdb_to_emotion_jsonl.py", line 85, in _find_first_existing
    raise FileNotFoundError(
FileNotFoundError: Could not find RAF-DB label file. Expected one of: list_patition_label.txt, list_partition_label.txt, EmoLabel/list_patition_label.txt, EmoLabel/list_partition_label.txt, basic/EmoLabel/list_patition_label.txt, basic/EmoLabel/list_partition_label.txt


CalledProcessError: Command '['/opt/tljh/user/envs/py310/bin/python', 'tools/convert_rafdb_to_emotion_jsonl.py', '--raf-root', '/home/jupyter-hault/EmotionCLIP-ReID/data/RAF-DB', '--output', '/home/jupyter-hault/EmotionCLIP-ReID/data/RAF-DB/manifest.jsonl', '--root-dir', '/home/jupyter-hault/EmotionCLIP-ReID/data/RAF-DB']' returned non-zero exit status 1.

## 3. Kiểm tra manifest và phân bố split

In [ ]:
import json
from collections import Counter

assert RAF_MANIFEST.exists(), f"Missing manifest: {RAF_MANIFEST}"
records = [json.loads(line) for line in RAF_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Total records:", len(records))
print("Split counts:", Counter(record["split"] for record in records))
print("Official split counts:", Counter(record.get("official_split") for record in records))
print("Emotion counts:", Counter(record["emotion"] for record in records))
print("Split protocol:", records[0].get("split_protocol"))
records[:3]

## 4. Preview ảnh mẫu

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_records = records[:8]
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, record in zip(axes.ravel(), sample_records):
    image_path = RAF_ROOT / record["image_path"]
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(f"{record['split']} / {record['emotion']}")
    ax.axis("off")
plt.tight_layout()

## 5. Train RAF-DB

In [ ]:
import subprocess
import sys

RAF_CONFIG = REPO / "configs" / "emotion" / "vit_b16_emotionclip_rafdb_quick.yml"
OUTPUT_DIR = REPO / "outputs" / "emotionclip_rafdb_quick"

train_cmd = [
    sys.executable,
    "train_emotionclip.py",
    "--config_file",
    str(RAF_CONFIG),
    "DATASETS.MANIFEST",
    str(RAF_MANIFEST),
    "DATASETS.ROOT_DIR",
    str(RAF_ROOT),
    "OUTPUT_DIR",
    str(OUTPUT_DIR),
]
subprocess.run(train_cmd, cwd=REPO, check=True)

## 6. Xem metrics tốt nhất hoặc epoch mới nhất

In [ ]:
import json
from pathlib import Path

metric_files = sorted(OUTPUT_DIR.glob("metrics_epoch_*.json"))
print("Metric files:", [path.name for path in metric_files[-5:]])
if metric_files:
    metrics = json.loads(metric_files[-1].read_text(encoding="utf-8"))
    for key in ["accuracy", "balanced_accuracy", "macro_f1", "avg_uncertainty", "avg_confidence", "ece", "uncertainty_risk_auc", "num_samples"]:
        print(key, metrics.get(key))
    print("Per-class F1:", metrics.get("per_class_f1"))

## 7. Infer một ảnh validation

In [ ]:
import subprocess
import sys

val_records = [record for record in records if record["split"] == "val"]
sample = val_records[0]
weight = OUTPUT_DIR / "best_emotionclip.pth"
image_path = RAF_ROOT / sample["image_path"]
print("Ground truth:", sample["emotion"])
print("Image:", image_path)
subprocess.run(
    [
        sys.executable,
        "infer_emotionclip.py",
        "--config_file",
        str(RAF_CONFIG),
        "--weight",
        str(weight),
        "--image",
        str(image_path),
    ],
    cwd=REPO,
    check=True,
)